In [1]:
!pip install -U mem0ai

In [2]:
from mem0 import Memory

In [3]:
CLASSIFY_INSTRUCTIONS = """
For every memory you extract, classify it into EXACTLY ONE of these three types:
[SEMANTIC], [EPISODIC], or [PROCEDURAL].

Put the classification tag at the VERY START of the memory.

==================================================
[SEMANTIC] — ENDURING FACTS ABOUT THE USER
==================================================

Semantic memory stores facts, attributes, characteristics, preferences, skills,
identity, relationships, or other relatively stable knowledge about the user.

It answers:

"What is true about the user?"

Examples:
"[SEMANTIC] User's name is Ajay"
"[SEMANTIC] User is a data professional"
"[SEMANTIC] User lives in Bangalore"
"[SEMANTIC] User knows Python"
"[SEMANTIC] User prefers concise explanations"
"[SEMANTIC] User works at Google"

A semantic memory should describe the USER'S ATTRIBUTE OR STATE itself,
not the event, activity, or experience through which that fact was established.

--------------------------------------------------
IMPORTANT: DO NOT CONFUSE CURRENT ACTIVITY WITH SEMANTIC MEMORY
--------------------------------------------------

The fact that something is happening NOW does NOT automatically make it
semantic.

Projects, courses, jobs being pursued, trips, applications, experiments,
conversations, tasks, and other time-bounded activities can remain EPISODIC
even when they are currently ongoing.

Example:

"User is building a Context Engineering course for CampusX."

If this refers to a specific course project that the user started at a particular
time, classify it as:

"[EPISODIC] User started building a Context Engineering course for CampusX
during the week of August 22, 2026."

Do NOT classify it as:

"[SEMANTIC] User is building a Context Engineering course for CampusX"

because the memory describes a specific project/activity with a temporal
beginning.

==================================================
[EPISODIC] — EVENTS, EXPERIENCES, AND TIME-BOUND ACTIVITIES
==================================================

Episodic memory stores specific events, experiences, activities, projects,
actions, interactions, decisions, or situations associated with a particular
time or period.

It answers:

"What happened?"
"What did the user do?"
"What was the user working on at that time?"
"What experience did the user have?"

An episodic memory may describe something that:
- happened once,
- happened repeatedly during a particular period,
- started at a particular time,
- is currently ongoing but has a defined temporal beginning,
- has a specific deadline or period,
- belongs to a particular project or situation.

Examples:

"[EPISODIC] User visited Japan in March 2024"
"[EPISODIC] User attended a data science workshop yesterday"
"[EPISODIC] User started building a Context Engineering course for CampusX
during the week of August 22, 2026"
"[EPISODIC] User began working on a memory system in August 2026"
"[EPISODIC] User interviewed with Microsoft last Friday"
"[EPISODIC] User completed a machine learning project last month"
"[EPISODIC] User discussed memory architecture with a colleague yesterday"

--------------------------------------------------
ONGOING PROJECT / ACTIVITY RULE
--------------------------------------------------

An activity does NOT become semantic merely because it is still ongoing.

If the memory describes a specific project, course, task, experiment,
application, trip, learning activity, or other time-bounded undertaking,
classify it as EPISODIC when it has a recognizable beginning or time period.

Example:

"User started building a Context Engineering course for CampusX in August 2026."

→ [EPISODIC]

"User is currently building a Context Engineering course for CampusX."

If this refers to the same specific course project:

→ [EPISODIC]

The fact that the project is still ongoing does not make it semantic.

==================================================
SEMANTIC VS EPISODIC — CORE DISTINCTION
==================================================

Use this test:

SEMANTIC:
"What is this person / what is true about this person?"

EPISODIC:
"What happened / what did this person do / what were they working on?"

Examples:

"User is a teacher."
→ [SEMANTIC]

"User started teaching at CampusX in 2025."
→ [EPISODIC]

"User lives in Bangalore."
→ [SEMANTIC]

"User moved to Bangalore in August 2026."
→ [EPISODIC] if the move itself is the memory being preserved.

However, if the purpose is to retain the resulting long-term fact:
"User now lives in Bangalore."
→ [SEMANTIC]

"User knows Python."
→ [SEMANTIC]

"User learned Python during a course in 2025."
→ [EPISODIC]

"User is currently working on a Context Engineering course."
→ [EPISODIC] when it refers to a specific course/project.

"User's profession is data science."
→ [SEMANTIC]

"User attended a data science conference yesterday."
→ [EPISODIC]

==================================================
EVENT → RESULTING FACT RULE
==================================================

Some events create or change a semantic fact.

In such cases, distinguish between the EVENT and the RESULTING FACT.

Event:
"[EPISODIC] User moved to Bangalore in August 2026."

Resulting enduring fact:
"[SEMANTIC] User lives in Bangalore."

Event:
"[EPISODIC] User joined Google in June 2026."

Resulting enduring fact:
"[SEMANTIC] User works at Google."

Event:
"[EPISODIC] User started learning Python in 2024."

Resulting enduring fact:
"[SEMANTIC] User knows Python."

Do not convert every event into semantic merely because it produced a
current state.

If the memory is about the EVENT, use EPISODIC.
If the memory is about the resulting ENDURING FACT, use SEMANTIC.

==================================================
[PROCEDURAL] — FUTURE BEHAVIOUR
==================================================

Procedural memory stores instructions, rules, or standing preferences that tell
the assistant HOW to behave in future interactions.

It answers:

"How should I behave?"

Examples:

"[PROCEDURAL] Always explain code step by step"
"[PROCEDURAL] Keep explanations concise"
"[PROCEDURAL] Never give unexplained code dumps"
"[PROCEDURAL] Use British English when responding"

==================================================
FINAL DECISION RULE
==================================================

1. Is it an instruction or rule about future behaviour?
   → [PROCEDURAL]

2. Is it an enduring fact, attribute, characteristic, preference, skill,
   identity, relationship, or stable state?
   → [SEMANTIC]

3. Is it a specific event, experience, project, activity, action, interaction,
   decision, or situation tied to a particular time/period or having a
   recognizable beginning?
   → [EPISODIC]

IMPORTANT:
Do NOT classify something as semantic simply because it is currently true.

Do NOT classify something as episodic simply because a date is mentioned.

Look at what the memory is fundamentally about:
- ENDURING FACT / ATTRIBUTE → SEMANTIC
- EVENT / EXPERIENCE / TIME-BOUND ACTIVITY → EPISODIC
- FUTURE BEHAVIOUR RULE → PROCEDURAL

==================================================
OUTPUT RULES
==================================================

- Use exactly ONE tag per memory.
- The tag must be the first thing in the memory.
- Put one space after the tag.
- Do not create a separate memory for the tag.
- Preserve important temporal information for episodic memories.
- Preserve the resulting enduring fact for semantic memories.
- When a specific ongoing project or activity has a temporal beginning,
  classify the project/activity as EPISODIC.
"""

In [4]:
config = {
    "llm":      {"provider": "ollama", "config": {"model": "gemma4:31b-cloud"}},
    "embedder": {"provider": "ollama", "config": {"model": "embeddinggemma:latest"}},
    "vector_store": {
        "provider": "qdrant",
        "config": {"embedding_model_dims": 768, "collection_name": "mem0_gemma3"},
    },
    "custom_instructions": CLASSIFY_INSTRUCTIONS,
    "version": "v1.1",
}

m = Memory.from_config(config)
print("Mem0 ready.")

[PostHog] Multiple active PostHog clients detected for the same project API key and host. Reuse one Posthog instance per app or process when possible to avoid competing background queues and missed shutdown flushes. Multiple clients are supported when intentional.


Mem0 ready.


In [5]:
m.delete_all(user_id="ajay")
print("Cleared Ajay's memories.")

Cleared Ajay's memories.


In [6]:
conversation = [
    {"role": "user", "content": "Hey, good morning! How's it going?"},
    {"role": "assistant", "content": "Morning! I'm doing well, thanks. How can I help?"},
    {"role": "user", "content": "Ugh, terrible traffic today, took me forever to get coffee. Anyway — I'm Ajay, I work as a data science instructor."},
    {"role": "assistant", "content": "Nice to meet you, Ajay!"},
    {"role": "user", "content": "I live in Delhi"},
    {"role": "assistant","content":"wow thats a nice city"},
    {"role": "user", "content": "By the way, what's the weather like this weekend, do you know?"},
    {"role": "assistant", "content": "I can't check live weather, but I can help with other things."},
    {"role": "user", "content": "No worries. Quick heads up — I'm allergic to peanuts, so please keep that in mind for any food suggestions."},
    {"role": "assistant", "content": "Got it, I'll avoid peanuts in any recommendations."},
    {"role": "user", "content": "Also I mostly code in Python and honestly prefer it over R for my work."},
    {"role": "assistant", "content": "Understood — Python it is."},
    {"role": "user", "content": "Haha my coffee just went cold while typing this. Oh well."},
    {"role": "assistant", "content": "Happens to the best of us!"},
    {"role": "user", "content": "One more thing about our sessions — please explain code step by step, not big dumps. It helps my students follow along."},
    {"role": "assistant", "content": "Sure, I'll walk through code step by step."},
    {"role": "user", "content": "Cool. Random question — what's 15% of 240? Just checking something quickly."},
    {"role": "assistant", "content": "15% of 240 is 36."},
    {"role": "user", "content": "Thanks! Last thing — I'm building a Context Engineering course for CampusX right now, started it last week."},
    {"role": "assistant", "content": "That sounds like a great project, good luck with it!"},
    {"role": "user", "content": "Alright, gotta run, talk later!"},
    {"role": "assistant", "content": "Take care, Ajay!"},
]

result1 = m.add(conversation, user_id="ajay")   # extracts durable facts, ignores the noise
print("Memory created.")
result1

E:\Anaconda\Lib\site-packages\spacy\cli\_util.py:23: DeprecationWarning: Importing 'parser.split_arg_string' is deprecated, it will only be available in 'shell_completion' in Click 9.0.
  from click.parser import split_arg_string
E:\Anaconda\Lib\site-packages\weasel\util\config.py:8: DeprecationWarning: Importing 'parser.split_arg_string' is deprecated, it will only be available in 'shell_completion' in Click 9.0.
  from click.parser import split_arg_string
fastembed not installed - BM25 keyword search disabled. Install it with: pip install "mem0ai[extras]"
E:\Anaconda\Lib\site-packages\mem0\vector_stores\qdrant.py:177: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  self.client.create_payload_index(


Memory created.


{'results': [{'id': '6bcb34f4-b3f6-4c63-8619-6b3fa4480aa9',
   'memory': "[SEMANTIC] User's name is Ajay and he works as a data science instructor",
   'event': 'ADD'},
  {'id': '32405842-187a-4a98-b098-365d44745669',
   'memory': '[SEMANTIC] User lives in Delhi',
   'event': 'ADD'},
  {'id': '4ecb8d48-10ca-4872-8b1f-22203fb32a9e',
   'memory': '[SEMANTIC] User is allergic to peanuts and requests that this be considered for any food suggestions',
   'event': 'ADD'},
  {'id': '772a6c2b-1517-4d78-bdfd-5cc34e7a77c9',
   'memory': '[SEMANTIC] User mostly codes in Python and prefers it over R for his work',
   'event': 'ADD'},
  {'id': 'e9a3cf3e-6014-4a8c-b306-b9e866276d57',
   'memory': "[PROCEDURAL] Assistant should explain code step by step rather than providing large code dumps to help User's students follow along",
   'event': 'ADD'},
  {'id': 'bf26b536-c265-43ca-83a2-5f6f8ac8a5da',
   'memory': '[EPISODIC] User started building a Context Engineering course for CampusX during the week 

In [7]:
for mem in m.get_all(filters={"user_id": "ajay"})["results"]:
    print("-", mem["memory"])

- [SEMANTIC] User lives in Delhi
- [SEMANTIC] User is allergic to peanuts and requests that this be considered for any food suggestions
- [SEMANTIC] User's name is Ajay and he works as a data science instructor
- [SEMANTIC] User mostly codes in Python and prefers it over R for his work
- [EPISODIC] User started building a Context Engineering course for CampusX during the week of August 22, 2026
- [PROCEDURAL] Assistant should explain code step by step rather than providing large code dumps to help User's students follow along


In [8]:
## CHANGES

# Some days later. Ajay's situation has changed. Notice: facts here CONTRADICT
# the old ones (moved city, switched language preference) -> Mem0 must update them.
update_conversation = [
    {"role": "user", "content": "Hey! Big update — I moved from Delhi to Bangalore last month."},
    {"role": "assistant", "content": "That's a big move, congrats!"},
    {"role": "user", "content": "Also, I've switched — I mostly work in R now, not Python anymore."},
    {"role": "assistant", "content": "Got it, R it is now."},
]

m.add(update_conversation, user_id="ajay")

{'results': [{'id': '2bfe9293-5062-4a20-9663-13a5aa32f463',
   'memory': '[EPISODIC] User moved from Delhi to Bangalore in July 2026',
   'event': 'ADD'},
  {'id': '91c82e30-4d4d-4abe-9f25-2a73e34f0bfd',
   'memory': '[SEMANTIC] User lives in Bangalore',
   'event': 'ADD'},
  {'id': '3ce153a9-a5f8-4318-bfae-c8712920cc0a',
   'memory': '[SEMANTIC] User now mostly works in R and has switched away from using Python as their primary language',
   'event': 'ADD'}]}

In [9]:
for mem in m.get_all(filters={"user_id": "ajay"})["results"]:
    print("-", mem["memory"])

- [EPISODIC] User moved from Delhi to Bangalore in July 2026
- [SEMANTIC] User lives in Delhi
- [SEMANTIC] User now mostly works in R and has switched away from using Python as their primary language
- [SEMANTIC] User is allergic to peanuts and requests that this be considered for any food suggestions
- [SEMANTIC] User's name is Ajay and he works as a data science instructor
- [SEMANTIC] User mostly codes in Python and prefers it over R for his work
- [SEMANTIC] User lives in Bangalore
- [EPISODIC] User started building a Context Engineering course for CampusX during the week of August 22, 2026
- [PROCEDURAL] Assistant should explain code step by step rather than providing large code dumps to help User's students follow along


#### SEPARATING  MEMORIES

In [24]:
from datetime import datetime

semantic = []   # SEMANTIC CONTAINER
procedural = []  # PROCEDURAL CONATAINER

for row in m.get_all(filters={"user_id": "ajay"})["results"]:   # ITERATING OVER MEORY 
    text = row["memory"]

    # Extract date and time
    dt = datetime.fromisoformat(row["created_at"])
    date = dt.strftime("%d-%m-%Y")
    time = dt.strftime("%I:%M:%S %p")

    if text.startswith("[SEMANTIC]"):
        semantic.append({
            "text": text.replace("[SEMANTIC]", "").strip(),
            "date": date,
            "time": time
        })

    elif text.startswith("[PROCEDURAL]"):
        procedural.append({
            "text": text.replace("[PROCEDURAL]", "").strip(),
            "date": date,
            "time": time
        })


print("SEMANTIC:")
for s in semantic:
    print(f"  - {s['text']}")
    print(f"    Date: {s['date']}")
    print(f"    Time: {s['time']}")

print("\nPROCEDURAL:")
for p in procedural:
    print(f"  - {p['text']}")
    print(f"    Date: {p['date']}")
    print(f"    Time: {p['time']}")

SEMANTIC:
  - User lives in Delhi
    Date: 29-08-2026
    Time: 07:25:09 AM
  - User now mostly works in R and has switched away from using Python as their primary language
    Date: 29-08-2026
    Time: 07:25:13 AM
  - User is allergic to peanuts and requests that this be considered for any food suggestions
    Date: 29-08-2026
    Time: 07:25:09 AM
  - User's name is Ajay and he works as a data science instructor
    Date: 29-08-2026
    Time: 07:25:09 AM
  - User mostly codes in Python and prefers it over R for his work
    Date: 29-08-2026
    Time: 07:25:09 AM
  - User lives in Bangalore
    Date: 29-08-2026
    Time: 07:25:13 AM

PROCEDURAL:
  - Assistant should explain code step by step rather than providing large code dumps to help User's students follow along
    Date: 29-08-2026
    Time: 07:25:09 AM


### REMOVING MULTIPLE MEMORIES OF SAME KIND.

In [25]:
import ollama


def deduplicate_semantic_memories(semantic):

    system_instruction = """
You are a semantic memory deduplication system.

Your task is to remove duplicate or conflicting memories.

Rules:
1. Identify memories that represent the SAME underlying fact.
2. If multiple memories represent the same fact, keep ONLY the latest one
   according to date and time.
3. If memories represent different facts, keep all of them.
4. Pay attention to changes over time.
   Example:
   "User lives in Delhi"
   "User moved to Bangalore"
   These represent the same fact (location), so keep only the latest one.
5. Do not merge unrelated memories.
6. Return ONLY the memory text.
7. Put each memory on a separate line.
8. Do not return JSON, dates, times, numbering, bullets, explanations,
   or any other extra text.
"""

    user_prompt = f"""
Semantic memories:

{semantic}

Return only the final memory texts, one memory per line.
"""

    response = ollama.generate(
        model="gemma4:31b-cloud",
        system=system_instruction,
        prompt=user_prompt
    )

    return response["response"].strip()

In [26]:
clean_semantic = deduplicate_semantic_memories(semantic)
print(clean_semantic)

User now mostly works in R and has switched away from using Python as their primary language
User is allergic to peanuts and requests that this be considered for any food suggestions
User's name is Ajay and he works as a data science instructor
User lives in Bangalore


In [27]:
clean_procedural = deduplicate_semantic_memories(procedural)
print(clean_procedural)

Assistant should explain code step by step rather than providing large code dumps to help User's students follow along


In [28]:
SYSTEM_INSTRUCTION = """You are a helpful personal assistant with memory of this user.

You will be given memory about the user in three parts:
- RULES: how this user wants you to behave. Always follow them.
- FACTS: stable truths about the user. Treat them as true and use them when relevant.
- RELEVANT EVENTS: things that happened before, retrieved for this question. Use them
  only if they help answer the current question; ignore them if they don't fit.

Rely on this memory instead of guessing. If the memory doesn't cover something, answer
normally and don't invent details about the user."""

In [29]:
SYSTEM_INSTRUCTION =  SYSTEM_INSTRUCTION + "\n\nRULES:\n" + clean_semantic  + "\n\nFACTS:\n" + clean_procedural

In [30]:
print(SYSTEM_INSTRUCTION)

You are a helpful personal assistant with memory of this user.

You will be given memory about the user in three parts:
- RULES: how this user wants you to behave. Always follow them.
- FACTS: stable truths about the user. Treat them as true and use them when relevant.
- RELEVANT EVENTS: things that happened before, retrieved for this question. Use them
  only if they help answer the current question; ignore them if they don't fit.

Rely on this memory instead of guessing. If the memory doesn't cover something, answer
normally and don't invent details about the user.

RULES:
User now mostly works in R and has switched away from using Python as their primary language
User is allergic to peanuts and requests that this be considered for any food suggestions
User's name is Ajay and he works as a data science instructor
User lives in Bangalore

FACTS:
Assistant should explain code step by step rather than providing large code dumps to help User's students follow along


In [31]:
# BASICS INFORMATION PART OF OUR SYSTEM INSTRUCTION

response = ollama.generate(
    model="gemma4:31b-cloud",
    system=SYSTEM_INSTRUCTION,
    prompt="tell me something about myself"
)

print(response["response"].strip())

You are Ajay, a data science instructor based in Bangalore. Professionally, you've shifted your primary focus to using R over Python. You also have a peanut allergy, which is an important consideration for any food recommendations.


In [32]:
for mem in m.get_all(filters={"user_id": "ajay"})["results"]:
    print("-", mem["memory"])

- [EPISODIC] User moved from Delhi to Bangalore in July 2026
- [SEMANTIC] User lives in Delhi
- [SEMANTIC] User now mostly works in R and has switched away from using Python as their primary language
- [SEMANTIC] User is allergic to peanuts and requests that this be considered for any food suggestions
- [SEMANTIC] User's name is Ajay and he works as a data science instructor
- [SEMANTIC] User mostly codes in Python and prefers it over R for his work
- [SEMANTIC] User lives in Bangalore
- [EPISODIC] User started building a Context Engineering course for CampusX during the week of August 22, 2026
- [PROCEDURAL] Assistant should explain code step by step rather than providing large code dumps to help User's students follow along


#### HANDLING EPISODIC MEMORY

In [33]:
user_query= "I am currenlty working on which project?"

In [34]:
def get_top_episodic_memories(user_query, user_id="ajay", threshold=0.30, top_k=2):

    result = m.search(
        user_query,
        filters={"user_id": user_id}
    )

    episodic = [
        r for r in result["results"]
        if r["memory"].startswith("[EPISODIC]")
        and r["score"] > threshold
    ]

    top_episodic = sorted(
        episodic,
        key=lambda x: x["score"],
        reverse=True
    )[:top_k]

    episodic_memories = [
        {
            "text": r["memory"],
            "date": r["created_at"].split("T")[0],
            "time": r["created_at"].split("T")[1].split("+")[0]
        }
        for r in top_episodic
    ]

    return episodic_memories

In [35]:
episodic = get_top_episodic_memories(user_query)

print(episodic)

[{'text': '[EPISODIC] User started building a Context Engineering course for CampusX during the week of August 22, 2026', 'date': '2026-08-29', 'time': '07:25:09.812614'}]


In [36]:
user_question= f"<question>{user_query}</question><retrieved_chunks>{episodic}</retrieved_chunks>"

In [37]:
user_question

"<question>I am currenlty working on which project?</question><retrieved_chunks>[{'text': '[EPISODIC] User started building a Context Engineering course for CampusX during the week of August 22, 2026', 'date': '2026-08-29', 'time': '07:25:09.812614'}]</retrieved_chunks>"

In [38]:
messages = [
    {
        "role": "system",
        "content": SYSTEM_INSTRUCTION
    },
    {
        "role": "user",
        "content": user_question
    }
]

response = ollama.chat(
    model="gemma4:31b-cloud",
    messages=messages
)

In [39]:
response

ChatResponse(model='gemma4:31b', created_at='2026-08-29T07:42:16.354523815Z', done=True, done_reason='stop', total_duration=231034934, load_duration=None, prompt_eval_count=329, prompt_eval_duration=None, eval_count=13, eval_duration=None, message=Message(role='assistant', content='You are currently building a Context Engineering course for CampusX.', thinking=None, images=None, tool_name=None, tool_calls=None))

In [40]:
print(response["message"]["content"])

You are currently building a Context Engineering course for CampusX.


## SINGLE FUNCTION

In [41]:
from datetime import datetime
import ollama


def answer_with_memory(user_query, user_id="ajay"):

    # --------------------------------------------------
    # 1. Get all memories
    # --------------------------------------------------

    memories = m.get_all(
        filters={"user_id": user_id}
    )["results"]


    # --------------------------------------------------
    # 2. Separate semantic and procedural memories
    # --------------------------------------------------

    semantic = []
    procedural = []

    for row in memories:

        text = row["memory"]

        dt = datetime.fromisoformat(
            row["created_at"]
        )

        date = dt.strftime("%d-%m-%Y")
        time = dt.strftime("%I:%M:%S %p")

        if text.startswith("[SEMANTIC]"):

            semantic.append({
                "text": text.replace("[SEMANTIC]", "").strip(),
                "date": date,
                "time": time
            })

        elif text.startswith("[PROCEDURAL]"):

            procedural.append({
                "text": text.replace("[PROCEDURAL]", "").strip(),
                "date": date,
                "time": time
            })


    # --------------------------------------------------
    # 3. Deduplicate semantic/procedural memories
    # --------------------------------------------------

    def deduplicate_memories(memory_list):

        if not memory_list:
            return ""

        system_instruction = """
You are a memory deduplication system.

Your task is to clean the given memories.

Rules:

1. Identify memories that represent the SAME underlying fact,
   preference, behavior, rule, state, or attribute.

2. If multiple memories represent the same underlying fact,
   keep ONLY the latest memory according to date and time.

3. If memories represent different facts, keep all of them.

4. Pay attention to changes over time.

Example:
"User lives in Delhi"
"User moved to Bangalore"

These represent the same fact (location), so keep only the latest one.

5. Do not merge unrelated memories.

6. Return ONLY the final memory text.

7. Put each memory on a separate line.

8. Do not return JSON, dates, times, numbering, bullets,
   explanations, or any other extra text.
"""

        user_prompt = f"""
Memories:

{memory_list}

Return only the final memories, one per line.
"""

        response = ollama.generate(
            model="gemma4:31b-cloud",
            system=system_instruction,
            prompt=user_prompt
        )

        return response["response"].strip()


    clean_semantic = deduplicate_memories(semantic)
    clean_procedural = deduplicate_memories(procedural)


    # --------------------------------------------------
    # 4. System instruction
    # --------------------------------------------------

    SYSTEM_INSTRUCTION = """
You are a helpful personal assistant with memory of this user.

You will be given memory about the user in three parts:

- RULES: how this user wants you to behave. Always follow them.
- FACTS: stable truths about the user. Treat them as true and use them
  when relevant.
- RELEVANT EVENTS: things that happened before, retrieved for this
  question. Use them only if they help answer the current question;
  ignore them if they don't fit.

Rely on this memory instead of guessing.

If the memory doesn't cover something, answer normally and don't
invent details about the user.
"""

    SYSTEM_INSTRUCTION = (
        SYSTEM_INSTRUCTION
        + "\n\nRULES:\n"
        + clean_procedural
        + "\n\nFACTS:\n"
        + clean_semantic
    )


    # --------------------------------------------------
    # 5. Retrieve relevant episodic memories
    # --------------------------------------------------

    result = m.search(
        user_query,
        filters={"user_id": user_id}
    )

    episodic = [
        r for r in result["results"]
        if r["memory"].startswith("[EPISODIC]")
        and r["score"] > 0.30
    ]


    # --------------------------------------------------
    # 6. Take top 2 episodic memories by score
    # --------------------------------------------------

    top_2_episodic = sorted(
        episodic,
        key=lambda x: x["score"],
        reverse=True
    )[:2]


    # --------------------------------------------------
    # 7. Extract text, date and time
    # --------------------------------------------------

    episodic = [
        {
            "text": r["memory"].replace("[EPISODIC]", "").strip(),
            "date": r["created_at"].split("T")[0],
            "time": r["created_at"].split("T")[1].split("+")[0]
        }
        for r in top_2_episodic
    ]


    # --------------------------------------------------
    # 8. Create final user prompt
    # --------------------------------------------------

    user_question = (
        f"<question>{user_query}</question>"
        f"<retrieved_chunks>{episodic}</retrieved_chunks>"
    )


    # --------------------------------------------------
    # 9. Call final LLM
    # --------------------------------------------------

    messages = [
        {
            "role": "system",
            "content": SYSTEM_INSTRUCTION
        },
        {
            "role": "user",
            "content": user_question
        }
    ]

    response = ollama.chat(
        model="gemma4:31b-cloud",
        messages=messages
    )


    # --------------------------------------------------
    # 10. Return final answer
    # --------------------------------------------------

    return response

In [42]:
answer = answer_with_memory(
    "I am currently working on which project?"
)
print(answer)

model='gemma4:31b' created_at='2026-08-29T07:44:06.455489013Z' done=True done_reason='stop' total_duration=446656740 load_duration=None prompt_eval_count=327 prompt_eval_duration=None eval_count=18 eval_duration=None message=Message(role='assistant', content='Based on recent events, you are currently building a Context Engineering course for CampusX.', thinking=None, images=None, tool_name=None, tool_calls=None)


In [43]:
answer = answer_with_memory(
    "What do you know about me ?"
)
print(answer)

model='gemma4:31b' created_at='2026-08-29T07:44:20.979806968Z' done=True done_reason='stop' total_duration=635345504 load_duration=None prompt_eval_count=262 prompt_eval_duration=None eval_count=144 eval_duration=None message=Message(role='assistant', content='Based on the information I have, here is what I know about you:\n\n*   **Professional Life:** Your name is Ajay, and you work as a data science instructor.\n*   **Technical Preferences:** While you may have used it in the past, you have switched away from Python as your primary language and now mostly work in R.\n*   **Location:** You live in Bangalore.\n*   **Health/Dietary Needs:** You have a peanut allergy, which I keep in mind for any food-related suggestions.\n*   **Teaching Style:** When explaining code, you prefer a step-by-step approach rather than large code dumps to help your students follow along more easily.', thinking=None, images=None, tool_name=None, tool_calls=None)
